In [1]:
# Configuracion
N_ITER = 4
CV = 2
RANDOM_STATE = 42
TARGET_CLASS = 1  # clase positiva; la clase nula es 0
TARGET_NAME = None  # None procesa todos los targets detectados
BASE_NAME = None
BALANCE_NAME = None



In [2]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    make_scorer,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import label_binarize

REPO_ROOT = Path('.').resolve()
INPUT_BASE = REPO_ROOT / 'data' / 'intermediate' / '05_seleccion_v2'
PRE_REPORTS_BASE = REPO_ROOT / 'reports' / '06_modelo_logistic_v2'
OUTPUT_BASE = REPO_ROOT / 'reports' / '10_tuning_logistic_v2'

BALANCE_METHODS = {
    'NONE': None,
    'SMOTE': SMOTE(random_state=RANDOM_STATE),
    'UNDER': RandomUnderSampler(random_state=RANDOM_STATE),
    'SMOTEENN': SMOTEENN(random_state=RANDOM_STATE),
}


def latest_file(pattern_base: Path, pattern: str) -> Path:
    files = sorted(pattern_base.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(f'No se encontro archivo con patron: {pattern_base / pattern}')
    return files[-1]


def latest_input_path() -> Path:
    candidates = sorted([p for p in INPUT_BASE.iterdir() if p.is_dir()])
    if not candidates:
        raise FileNotFoundError(f'No hay subdirectorios en {INPUT_BASE}')
    return candidates[-1]


def choose_candidates(summary_csv: Path) -> list[dict]:
    df = pd.read_csv(summary_csv)
    req = {'target', 'modelo', 'balanceo', 'cv_recall_target', 'cv_macro_f1'}
    miss = req - set(df.columns)
    if miss:
        raise ValueError(f'Faltan columnas en resumen: {sorted(miss)}')

    d = df.dropna(subset=['target', 'cv_recall_target', 'cv_macro_f1']).copy()
    if TARGET_NAME is not None:
        d = d[d['target'].astype(str) == str(TARGET_NAME)]
    if d.empty:
        raise ValueError('No hay filas validas con target, cv_recall_target y cv_macro_f1')

    candidates = []
    for target_name, df_target in d.groupby('target', sort=True):
        best = df_target.sort_values(['cv_recall_target', 'cv_macro_f1'], ascending=False).iloc[0]
        candidates.append({
            'tag': f"best_{target_name}",
            'target': str(target_name),
            'base_name': str(best['modelo']),
            'balance_name': str(best['balanceo']),
            'cv_recall_target': float(best['cv_recall_target']),
            'cv_macro_f1': float(best['cv_macro_f1']),
        })
    return candidates


def resolve_target_class(y: pd.Series, target: int | None) -> int:
    classes = list(pd.Series(y).unique())
    if target is None:
        return 1 if 1 in classes else int(classes[-1])
    if target in classes:
        return int(target)
    if str(target) in [str(c) for c in classes]:
        for c in classes:
            if str(c) == str(target):
                return int(c)
    raise ValueError(f'TARGET_CLASS {target} no esta en clases disponibles: {classes}')


def build_pipeline(balancer) -> Pipeline:
    steps = []
    steps.append(('impute', SimpleImputer(strategy='median')))
    if balancer is not None:
        steps.append(('balance', clone(balancer)))
    steps.append(('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)))
    return Pipeline(steps)


def save_eval_pdf(y_true, y_pred, y_proba, class_order, out_pdf: Path, title_prefix: str):
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred, labels=class_order)

    with PdfPages(out_pdf) as pdf:
        ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order).plot(cmap='Blues', values_format='d')
        plt.title(f'{title_prefix} - Matriz de Confusion')
        pdf.savefig(); plt.close()

        if np.unique(y_true).size >= 2:
            y_bin = label_binarize(y_true, classes=class_order)
            proba_for_curves = y_proba
            if y_bin.shape[1] == 1:
                y_bin = np.hstack([1 - y_bin, y_bin])
                if proba_for_curves.shape[1] == 1:
                    proba_for_curves = np.hstack([1 - proba_for_curves, proba_for_curves])

            plt.figure()
            for i, clase in enumerate(class_order):
                fpr, tpr, _ = roc_curve(y_bin[:, i], proba_for_curves[:, i])
                roc_auc = auc(fpr, tpr)
                plt.plot(fpr, tpr, label=f'Clase {clase} (AUC={roc_auc:.2f})')
            plt.plot([0, 1], [0, 1], 'k--')
            plt.title(f'{title_prefix} - Curvas ROC por clase')
            plt.xlabel('FPR')
            plt.ylabel('TPR')
            plt.legend()
            pdf.savefig(); plt.close()

            plt.figure()
            for i, clase in enumerate(class_order):
                precision, recall, _ = precision_recall_curve(y_bin[:, i], proba_for_curves[:, i])
                pr_auc = average_precision_score(y_bin[:, i], proba_for_curves[:, i])
                plt.plot(recall, precision, label=f'Clase {clase} (PR AUC={pr_auc:.2f})')
            plt.title(f'{title_prefix} - Curvas Precision-Recall por clase')
            plt.xlabel('Recall')
            plt.ylabel('Precision')
            plt.legend()
            pdf.savefig(); plt.close()



In [3]:
summary_csv = latest_file(PRE_REPORTS_BASE, '**/resumen_modelos_logistic_regression_global.csv')
input_path = latest_input_path()

if BASE_NAME and BALANCE_NAME:
    if TARGET_NAME is None:
        raise ValueError('Para modo manual debes definir TARGET_NAME, BASE_NAME y BALANCE_NAME')
    candidates = [{
        'tag': f'manual_{TARGET_NAME}',
        'target': TARGET_NAME,
        'base_name': BASE_NAME,
        'balance_name': BALANCE_NAME,
        'cv_recall_target': None,
        'cv_macro_f1': None,
    }]
else:
    candidates = choose_candidates(summary_csv)

run_id = datetime.today().strftime('%Y%m%d')
output_dir = OUTPUT_BASE / run_id / f"Logistic_tuning_v2_{datetime.today().strftime('%Y-%m-%d')}"
output_dir.mkdir(parents=True, exist_ok=True)

all_summary = []

for cand in candidates:
    tag = cand['tag']
    target_name = str(cand['target'])
    base_name = cand['base_name']
    balance_name = cand['balance_name']
    target_input_path = input_path / target_name

    if balance_name not in BALANCE_METHODS:
        raise ValueError(f'Balanceo no soportado: {balance_name}')
    if not target_input_path.exists():
        raise FileNotFoundError(f'No existe input para target {target_name}: {target_input_path}')

    print(f"\n=== Candidato: {tag} | {target_name} | {base_name} [{balance_name}] ===")

    X_train = pd.read_parquet(target_input_path / f'X_train_{base_name}.parquet')
    X_test = pd.read_parquet(target_input_path / f'X_test_{base_name}.parquet')
    X_train = X_train.astype('float32')
    X_test = X_test.astype('float32')
    y_train = pd.read_parquet(target_input_path / f'y_train_{base_name}.parquet').squeeze()
    y_test = pd.read_parquet(target_input_path / f'y_test_{base_name}.parquet').squeeze()

    target_class = resolve_target_class(y_train, TARGET_CLASS)

    balancer = BALANCE_METHODS[balance_name]
    pipeline = build_pipeline(balancer)

    scorers = {
        'recall_target': make_scorer(recall_score, labels=[target_class], average='macro', zero_division=0),
        'f1_macro': 'f1_macro',
    }

    param_distributions = {
        'model__C': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
        'model__solver': ['lbfgs', 'saga'],
        'model__penalty': ['l2'],
        'model__class_weight': [None, 'balanced'],
        'model__max_iter': [500, 1000, 1500],
    }

    cv = StratifiedKFold(n_splits=CV, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=N_ITER,
        scoring=scorers,
        refit='recall_target',
        cv=cv,
        n_jobs=1,
        random_state=RANDOM_STATE,
        verbose=1,
        return_train_score=False,
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    y_pred_test = best_model.predict(X_test)
    y_pred_train = best_model.predict(X_train)
    y_proba = best_model.predict_proba(X_test)
    class_order = best_model.named_steps['model'].classes_

    cand_dir = output_dir / target_name / tag
    cand_dir.mkdir(parents=True, exist_ok=True)

    pdf_path = cand_dir / 'reporte_tuning_logistic.pdf'
    title = f'{target_name} - {tag} - {base_name} [{balance_name}]'
    save_eval_pdf(y_test, y_pred_test, y_proba, class_order, pdf_path, title)

    cv_results = pd.DataFrame(search.cv_results_).sort_values('rank_test_recall_target')
    cv_results.to_csv(cand_dir / 'cv_results_logistic_tuning.csv', index=False)

    summary = {
        'target': target_name,
        'candidate_tag': tag,
        'base_name': base_name,
        'balanceo': balance_name,
        'summary_csv_source': str(summary_csv),
        'input_path': str(target_input_path),
        'target_class_original': int(target_class),
        'selected_cv_recall_target': cand['cv_recall_target'],
        'selected_cv_macro_f1': cand['cv_macro_f1'],
        'cv_best_recall_target': float(search.best_score_),
        'cv_best_f1_macro': float(cv_results.iloc[0]['mean_test_f1_macro']),
        'test_accuracy': float(accuracy_score(y_test, y_pred_test)),
        'test_f1_macro': float(f1_score(y_test, y_pred_test, average='macro', zero_division=0)),
        'train_f1_macro': float(f1_score(y_train, y_pred_train, average='macro', zero_division=0)),
        'pdf_report': str(pdf_path),
    }

    pd.DataFrame([summary]).to_csv(cand_dir / 'resumen_tuning_logistic.csv', index=False)
    with open(cand_dir / 'best_params_logistic.json', 'w', encoding='utf-8') as f:
        json.dump(search.best_params_, f, indent=2)

    report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)).T
    report_test.to_csv(cand_dir / 'classification_report_test_logistic.csv')

    all_summary.append(summary)
    print(json.dumps(summary, indent=2))

pd.DataFrame(all_summary).to_csv(output_dir / 'resumen_tuning_logistic_all_candidates.csv', index=False)
print(f'Salida: {output_dir}')




=== Candidato: best_1_vs_resto | 1_vs_resto | Robust_ORIGINAL_ALL [SMOTEENN] ===
Fitting 2 folds for each of 4 candidates, totalling 8 fits


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
   

{
  "target": "1_vs_resto",
  "candidate_tag": "best_1_vs_resto",
  "base_name": "Robust_ORIGINAL_ALL",
  "balanceo": "SMOTEENN",
  "summary_csv_source": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\reports\\06_modelo_logistic_v2\\20260401\\Logistic_v2_01_2026-04-01\\resumen_modelos_logistic_regression_global.csv",
  "input_path": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\data\\intermediate\\05_seleccion_v2\\05_2026_03_31\\1_vs_resto",
  "target_class_original": 1,
  "selected_cv_recall_target": 0.9121488121054409,
  "selected_cv_macro_f1": 0.4712525607352349,
  "cv_best_recall_target": 0.9144378584164619,
  "cv_best_f1_macro": 0.4670950648906979,
  "test_accuracy": 0.5491623400952738,
  "test_f1_macro": 0.4733879538834461,
  "train_f1_macro": 0.47318686833757356,
  "pdf_report": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\reports\\10_tuning_logistic_v2\\20260413\\Logistic_tuning_v2_2026-04-13\\1_vs_resto